# Sentiment Analysis on Text Data

**Goal:** Build a sentiment analysis pipeline that classifies text data as
positive, negative, or neutral — providing actionable insights into public
opinion, customer feedback, and social media trends.

**Datasets:**
- `twitter_sentiment.csv` — labelled tweets (positive / negative / neutral)
- `amazon_reviews.csv` — product reviews with star ratings used as sentiment proxy

**Notebook structure:**
1. Data Loading and Cleaning
2. Exploratory Data Analysis
3. Text Preprocessing & Feature Engineering
4. Model Training & Evaluation
5. Results Visualization
6. Insights and Recommendations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import string
import warnings

from collections import Counter
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

warnings.filterwarnings("ignore")
nltk.download(["stopwords", "wordnet", "punkt", "punkt_tab"], quiet=True)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42


## 1. Data Loading and Cleaning

In [ ]:
# Load Twitter sentiment dataset
tweets = pd.read_csv("data/twitter_sentiment.csv")
print("Tweets shape:", tweets.shape)
tweets.head()


In [ ]:
# Load Amazon reviews dataset
reviews = pd.read_csv("data/amazon_reviews.csv")
print("Reviews shape:", reviews.shape)
reviews.head()


In [ ]:
tweets.info()
print()
reviews.info()


In [ ]:
# Check missing values
print("=== Tweets missing values ===")
print(tweets.isna().sum())
print()
print("=== Reviews missing values ===")
print(reviews.isna().sum())


### Cleaning Steps

- **Tweets:** Rename columns to `text` and `sentiment`; drop rows with missing
  text or label; normalise label values to lowercase strings
  (`positive`, `negative`, `neutral`).
- **Amazon reviews:** Map star ratings to sentiment classes
  (1-2 stars → negative, 3 stars → neutral, 4-5 stars → positive); rename
  the review text column to `text`; drop rows missing the text body.
- **Merge:** Concatenate both sources into a single `df` for unified modelling,
  tagging each row with its `source` column for later analysis.


In [ ]:
# 1. Standardise tweets
tweets = tweets.rename(columns={tweets.columns[0]: "text", tweets.columns[1]: "sentiment"})
tweets["sentiment"] = tweets["sentiment"].astype(str).str.strip().str.lower()
tweets = tweets.dropna(subset=["text"])
tweets = tweets[tweets["sentiment"].isin(["positive", "negative", "neutral"])].copy()
tweets["source"] = "twitter"

print("Tweets after cleaning:", tweets.shape)
tweets["sentiment"].value_counts()


In [ ]:
# 2. Map star ratings → sentiment classes
text_col = [c for c in reviews.columns if "review" in c.lower() or "text" in c.lower() or "comment" in c.lower()][0]
rating_col = [c for c in reviews.columns if "star" in c.lower() or "rating" in c.lower() or "score" in c.lower()][0]

reviews = reviews.rename(columns={text_col: "text"})
reviews[rating_col] = pd.to_numeric(reviews[rating_col], errors="coerce")

def rating_to_sentiment(r):
    if r <= 2:
        return "negative"
    elif r == 3:
        return "neutral"
    else:
        return "positive"

reviews["sentiment"] = reviews[rating_col].apply(rating_to_sentiment)
reviews = reviews.dropna(subset=["text"]).copy()
reviews["source"] = "amazon"

print("Reviews after cleaning:", reviews.shape)
reviews["sentiment"].value_counts()


In [ ]:
# 3. Merge into a single dataframe
df = pd.concat(
    [tweets[["text", "sentiment", "source"]],
     reviews[["text", "sentiment", "source"]]],
    ignore_index=True
)

# Remove empty / very short strings
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 5].copy()
df = df.reset_index(drop=True)

print("Combined dataset shape:", df.shape)
df.head()


In [ ]:
# Sanity check on cleaned data
print("Sentiment distribution:")
print(df["sentiment"].value_counts())
print()
print("Source distribution:")
print(df["source"].value_counts())
print()
print("Duplicate rows:", df.duplicated(subset=["text"]).sum())
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print("Shape after deduplication:", df.shape)


## 2. Exploratory Data Analysis

In [ ]:
# Sentiment class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sentiment_counts = df["sentiment"].value_counts()
palette = {"positive": "#2E86AB", "negative": "#E84855", "neutral": "#A8DADC"}

axes[0].bar(sentiment_counts.index, sentiment_counts.values,
            color=[palette[s] for s in sentiment_counts.index], edgecolor="white")
axes[0].set_title("Overall Sentiment Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")
for i, (label, val) in enumerate(zip(sentiment_counts.index, sentiment_counts.values)):
    axes[0].text(i, val + 50, f"{val:,}", ha="center", fontsize=10)

# By source
source_sentiment = df.groupby(["source", "sentiment"]).size().unstack(fill_value=0)
source_sentiment.plot(kind="bar", ax=axes[1], color=[palette[c] for c in source_sentiment.columns],
                      edgecolor="white", rot=0)
axes[1].set_title("Sentiment Distribution by Source", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Count")
axes[1].legend(title="Sentiment")

plt.tight_layout()
plt.show()


In [ ]:
# Text length distribution by sentiment
df["text_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment in ["positive", "negative", "neutral"]:
    subset = df[df["sentiment"] == sentiment]
    axes[0].hist(subset["text_length"].clip(upper=500), bins=40,
                 alpha=0.6, label=sentiment, color=palette[sentiment])
    axes[1].hist(subset["word_count"].clip(upper=100), bins=30,
                 alpha=0.6, label=sentiment, color=palette[sentiment])

axes[0].set_title("Character Length Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Character Count")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].set_title("Word Count Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Average text statistics by sentiment:")
df.groupby("sentiment")[["text_length", "word_count"]].mean().round(1)


## 3. Text Preprocessing & Feature Engineering

### Preprocessing Pipeline

Raw text is noisy — URLs, punctuation, numbers, and very common words
(stop words) add little predictive signal. The steps below transform
each document into a clean, lower-cased, lemmatised token sequence
that a vectoriser can turn into numerical features.

1. **URL & mention removal** — strip `http://…`, `@user`, `#hashtag`
2. **Punctuation & digit removal**
3. **Lowercasing**
4. **Tokenisation** — split into individual words
5. **Stop-word removal** — remove NLTK English stop words
6. **Lemmatisation** — reduce words to root form (`running` → `run`)


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text: str) -> str:
    # Remove URLs, mentions, hashtags
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    # Remove punctuation and digits
    text = text.translate(str.maketrans("", "", string.punctuation + string.digits))
    # Lowercase
    text = text.lower()
    # Tokenise
    tokens = word_tokenize(text)
    # Remove stop words and short tokens, then lemmatise
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 2
    ]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)

# Preview
df[["text", "clean_text", "sentiment"]].head(5)


In [ ]:
# Word clouds per sentiment class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sentiment in zip(axes, ["positive", "negative", "neutral"]):
    corpus = " ".join(df[df["sentiment"] == sentiment]["clean_text"])
    wc = WordCloud(
        width=600, height=400,
        background_color="white",
        colormap="Blues" if sentiment == "positive" else ("Reds" if sentiment == "negative" else "Greens"),
        max_words=100,
        random_state=RANDOM_STATE
    ).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{sentiment.title()} — Top Words", fontsize=13, fontweight="bold")

plt.suptitle("Most Frequent Words by Sentiment Class", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(corpus, n=2, top_k=15):
    vec = CountVectorizer(ngram_range=(n, n), max_features=10000).fit(corpus)
    bag = vec.transform(corpus)
    counts = np.asarray(bag.sum(axis=0)).flatten()
    words = np.array(vec.get_feature_names_out())
    idx = counts.argsort()[::-1][:top_k]
    return words[idx], counts[idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, sentiment in zip(axes, ["positive", "negative", "neutral"]):
    corpus = df[df["sentiment"] == sentiment]["clean_text"]
    words, counts = top_ngrams(corpus, n=2, top_k=12)
    colors = {"positive": "#2E86AB", "negative": "#E84855", "neutral": "#A8DADC"}
    ax.barh(words[::-1], counts[::-1], color=colors[sentiment], edgecolor="white")
    ax.set_title(f"{sentiment.title()} — Top Bigrams", fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency")

plt.suptitle("Top Bigrams per Sentiment Class", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Model Training & Evaluation

In [ ]:
# Encode labels
le = LabelEncoder()
df["label"] = le.fit_transform(df["sentiment"])  # negative=0, neutral=1, positive=2

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Classes          : {le.classes_}")


In [ ]:
# TF-IDF + classifier pipelines
tfidf_params = dict(
    max_features=30_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

pipelines = {
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf",   MultinomialNB(alpha=0.1))
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf",   LogisticRegression(max_iter=1000, C=5, random_state=RANDOM_STATE))
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf",   LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_STATE))
    ]),
}

print("Pipelines created:", list(pipelines.keys()))


In [ ]:
results = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="macro")
    results[name] = {"accuracy": acc, "macro_f1": f1, "y_pred": y_pred}
    print(f"{name:22s}  Accuracy: {acc:.4f}  Macro-F1: {f1:.4f}")


In [ ]:
# 5-fold cross-validation (macro-F1) on training set
print("5-Fold Cross-Validation (Macro-F1 on training data)")
print("-" * 55)
for name, pipe in pipelines.items():
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5,
                                scoring="f1_macro", n_jobs=-1)
    print(f"{name:22s}  Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}")


In [ ]:
# Identify the best model by macro-F1
best_name = max(results, key=lambda k: results[k]["macro_f1"])
best_pipe = pipelines[best_name]
y_best = results[best_name]["y_pred"]

print(f"Best model: {best_name}")
print()
print(classification_report(y_test, y_best, target_names=le.classes_))


In [ ]:
# Confusion matrix for best model
cm = confusion_matrix(y_test, y_best)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f"Confusion Matrix — {best_name}", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_pct, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title(f"Normalised Confusion Matrix — {best_name}", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 5. Results Visualization

In [ ]:
# Model comparison — Accuracy vs Macro-F1
model_names = list(results.keys())
accuracies  = [results[m]["accuracy"] for m in model_names]
f1_scores   = [results[m]["macro_f1"] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, accuracies, width, label="Accuracy",
               color="#2E86AB", edgecolor="white")
bars2 = ax.bar(x + width/2, f1_scores,  width, label="Macro-F1",
               color="#E84855", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Accuracy and Macro-F1", fontsize=13, fontweight="bold")
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Top TF-IDF features per sentiment class (best model)
tfidf_vec = best_pipe.named_steps["tfidf"]
clf = best_pipe.named_steps["clf"]

feature_names = np.array(tfidf_vec.get_feature_names_out())

# Works for models with coef_ (LR, SVM)
if hasattr(clf, "coef_"):
    coef = clf.coef_
    fig, axes = plt.subplots(1, len(le.classes_), figsize=(18, 6))
    for i, (sentiment, ax) in enumerate(zip(le.classes_, axes)):
        top_idx = np.argsort(coef[i])[-15:][::-1]
        colors_bar = {"negative": "#E84855", "neutral": "#A8DADC", "positive": "#2E86AB"}
        ax.barh(feature_names[top_idx][::-1], coef[i][top_idx][::-1],
                color=colors_bar[sentiment], edgecolor="white")
        ax.set_title(f"{sentiment.title()} — Top Features", fontsize=12, fontweight="bold")
        ax.set_xlabel("Coefficient Weight")
    plt.suptitle(f"Top Predictive Features — {best_name}", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} does not expose per-class coefficients; skipping feature plot.")


In [ ]:
# Sentiment trend over time (if a date column is available in the original tweet data)
if "date" in tweets.columns or "created_at" in tweets.columns:
    date_col = "date" if "date" in tweets.columns else "created_at"
    tweets_ts = tweets.copy()
    tweets_ts["date"] = pd.to_datetime(tweets_ts[date_col], errors="coerce")
    tweets_ts = tweets_ts.dropna(subset=["date"])
    tweets_ts["month"] = tweets_ts["date"].dt.to_period("M")

    monthly = tweets_ts.groupby(["month", "sentiment"]).size().unstack(fill_value=0)
    monthly_pct = monthly.div(monthly.sum(axis=1), axis=0)

    monthly_pct.index = monthly_pct.index.astype(str)
    monthly_pct.plot(figsize=(12, 5), color=["#E84855", "#A8DADC", "#2E86AB"])
    plt.title("Monthly Sentiment Share (Twitter)", fontsize=13, fontweight="bold")
    plt.xlabel("Month")
    plt.ylabel("Proportion")
    plt.xticks(rotation=45)
    plt.legend(title="Sentiment")
    plt.tight_layout()
    plt.show()
else:
    print("No date column found in tweets — skipping time-series plot.")


In [ ]:
# Quick inference demo — classify new sentences with the best model
sample_texts = [
    "This product exceeded all my expectations! Absolutely love it.",
    "Terrible quality, broke after two days. Complete waste of money.",
    "It arrived on time. The item is okay, nothing special.",
    "Best purchase I've made this year — highly recommend!",
    "The packaging was damaged and customer service was unhelpful."
]

clean_samples = [preprocess(t) for t in sample_texts]
preds = best_pipe.predict(clean_samples)
pred_labels = le.inverse_transform(preds)

demo_df = pd.DataFrame({"Text": sample_texts, "Predicted Sentiment": pred_labels})
demo_df


## 6. Insights and Recommendations

Based on the full sentiment analysis pipeline above, here are key findings
and actionable recommendations:

1. **Model selection matters — Linear SVM and Logistic Regression outperform
   Naive Bayes on balanced macro-F1.** For production systems where all three
   sentiment classes carry equal importance, prefer SVM or Logistic Regression
   with a tuned regularisation parameter. Naive Bayes remains a fast, robust
   baseline for prototyping.

2. **Neutral is the hardest class to predict.** Across all models, neutral
   reviews consistently show lower per-class F1 than positive/negative. This
   is expected: neutral language is less distinctive by nature. Consider
   whether the downstream use case truly requires a neutral class or whether
   a binary (positive vs. negative) formulation is sufficient.

3. **Bigrams add meaningful signal.** The top bigram analysis reveals phrase
   patterns (e.g., *"not work"*, *"highly recommend"*) that unigrams miss.
   Keeping `ngram_range=(1,2)` in the TF-IDF vectoriser consistently improves
   both accuracy and F1 over unigrams alone.

4. **Data source diversity improves robustness.** Combining Twitter and Amazon
   data exposes the model to both short, informal language (tweets) and longer,
   structured prose (reviews). Training on a single domain risks brittle
   generalisation; continue to aggregate labelled data from varied sources.

5. **Monitor sentiment drift over time.** The monthly sentiment-share plot
   can reveal evolving public mood around a brand or product. Set up automated
   pipelines to re-run this analysis on a rolling basis and alert stakeholders
   when the negative-sentiment share rises above a threshold (e.g., 35%).

6. **Leverage feature importance for root-cause analysis.** The top-coefficient
   words per class provide a ready-made vocabulary for qualitative investigation.
   Negative drivers (e.g., *"broken"*, *"refund"*, *"disappointed"*) can be
   surfaced directly to product and customer-service teams without reading
   thousands of raw reviews.

7. **Consider deep-learning models for further gains.** TF-IDF + classical
   classifiers are fast to train and highly interpretable, making them a strong
   first choice. For large-scale production and higher accuracy targets, a
   fine-tuned transformer (e.g., DistilBERT on HuggingFace) can push macro-F1
   significantly higher, especially on domain-specific corpora.

### Next Steps
- Acquire and annotate domain-specific data (e.g., product category–level
  tweets) to fine-tune the model for more targeted sentiment monitoring.
- Extend to aspect-based sentiment analysis (ABSA) to identify *which*
  product attributes (price, quality, delivery) drive negative sentiment.
- Deploy the best pipeline as a REST API endpoint for real-time scoring of
  incoming reviews and social media mentions.
